# Product Catalog Normalization at Scale

Extract and normalize product attributes from messy supplier data. This example shows production features: cost control, checkpoint/resume, per-field normalization routing, and export.

**What you'll learn:** `CompositeNormalization` (per-field routing), `max_cost_usd`, `checkpoint_dir`, DataFrame export.

**Estimated API cost:** ~$0.03

In [ ]:
# pip install "catchfly[openai,export]"
# export OPENAI_API_KEY="sk-..."

## Define your schema

When you know what fields you need, skip discovery and provide a Pydantic model:

In [ ]:
from pydantic import BaseModel, Field


class ProductListing(BaseModel):
    product_name: str = Field(description="Full product name")
    brand: str = Field(description="Manufacturer brand name")
    category: str = Field(description="Product category, e.g. Smartphones, Audio, Laptops")
    rating: float | None = Field(default=None, description="Rating normalized to 0-10 scale")
    price: float | None = Field(default=None, description="Price in USD")
    pros: list[str] = Field(default_factory=list, description="Positive attributes")
    cons: list[str] = Field(default_factory=list, description="Negative attributes")

## Configure per-field normalization

Different fields need different strategies. `CompositeNormalization` routes each field to the right one.

This is the **Level 2** approach — you pick which fields to normalize and which strategy each gets. For simpler cases, `Pipeline.quick()` auto-selects fields via `StatisticalFieldSelector` with zero configuration.

In [ ]:
from catchfly.normalization import (
    CompositeNormalization,
    DictionaryNormalization,
    LLMCanonicalization,
)

normalization = CompositeNormalization(
    field_strategies={
        # Known brands: fast dictionary lookup, zero LLM cost
        "brand": DictionaryNormalization(
            mapping={
                "samsung": "Samsung", "SAMSUNG": "Samsung",
                "asus": "ASUS", "Asus": "ASUS",
                "lg": "LG", "Lg": "LG",
                "apple": "Apple", "APPLE": "Apple",
            },
            case_insensitive=True,
            passthrough_unmapped=True,
        ),
        # Categories: LLM groups semantically similar labels
        "category": LLMCanonicalization(model="gpt-5.4-mini"),
        # Pros/cons: LLM groups variant phrasings
        "pros": LLMCanonicalization(model="gpt-5.4-mini"),
        "cons": LLMCanonicalization(model="gpt-5.4-mini"),
    },
    # Fields not listed (rating, price) are not normalized
)

## Build the pipeline

In [ ]:
from catchfly import Pipeline
from catchfly.extraction import LLMDirectExtraction
from catchfly.demo import load_samples

docs = load_samples("product_reviews")

pipeline = Pipeline(
    extraction=LLMDirectExtraction(
        model="gpt-5.4-mini",
        on_error="collect",
    ),
    normalization=normalization,
    verbose=True,
)

## Estimate cost before running

In [ ]:
estimate = pipeline.estimate_cost(docs)
print(estimate)

## Run with cost budget and checkpoints

In [ ]:
results = pipeline.run(
    docs,
    schema=ProductListing,
    normalize_fields=["brand", "category", "pros", "cons"],
    max_cost_usd=2.0,
    checkpoint_dir="./checkpoints/product_catalog",
)

print(f"Records: {len(results.records)}, Errors: {len(results.errors)}")
print(f"Cost: ${results.report.total_cost_usd:.4f}")

## Inspect extracted records

In [ ]:
for record in results.records[:5]:
    print(f"{record.brand} {record.product_name}: ${record.price}, {record.rating}/10")
    print(f"  Pros: {record.pros}")
    print(f"  Cons: {record.cons}")
    print()

## Normalization results

### Category grouping

In [ ]:
cat_norm = results.normalizations.get("category")
if cat_norm:
    print("Category groups:")
    for canonical, members in cat_norm.clusters.items():
        print(f"  {canonical}: {members}")

### Pros grouping

In [ ]:
pros_norm = results.normalizations.get("pros")
if pros_norm:
    print("Pros groups (multi-member only):")
    for canonical, members in pros_norm.clusters.items():
        if len(members) > 1:
            print(f"  {canonical}: {members}")

## Export to DataFrame

In [ ]:
df = results.to_dataframe()
df[["product_name", "brand", "category", "rating", "price"]]

In [ ]:
# Save to Parquet
# df.to_parquet("product_catalog.parquet")

# Or CSV
# df.to_csv("product_catalog.csv", index=False)

## Scaling to thousands of products

For large catalogs, use glob patterns and checkpoints:

```python
results = pipeline.run(
    documents=["data/products/*.txt"],  # glob pattern
    schema=ProductListing,
    normalize_fields=["brand", "category", "pros"],
    max_cost_usd=10.0,
    checkpoint_dir="./checkpoints/large_catalog",
)
```

If interrupted, re-running with the same `checkpoint_dir` resumes from where it left off.

## Simpler alternative: auto field selection

If you don't need per-field strategy routing, `Pipeline.quick()` handles everything — including deciding which fields to normalize:

```python
pipeline = Pipeline.quick(model="gpt-5.4-mini")
results = pipeline.run(docs, schema=ProductListing)
# StatisticalFieldSelector auto-picks categorical fields (brand, category, pros, cons)
# and skips free-text fields (product_name) and numerics (rating, price)
```